In [8]:
import pandas as pd
import numpy as np
ctd = pd.read_csv("../Data/downcast_all.csv")
print(ctd.head())

/var/folders/xn/9gqx3sxx3s32k1ttvhjc4fch0000gn/T/ipykernel_77701/975230851.py:3: DtypeWarning: Columns (2,3,7,8,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  ctd = pd.read_csv("../Data/downcast_all.csv")


   PROJECT   STUDY ORD_OCC EVENT_NUM    CAST_ID         DATE_TIME_UTC  \
0  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:05:46Z   
1  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:04Z   
2  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:10Z   
3  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:11Z   
4  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:14Z   

          DATE_TIME_PST LAT_DEC LON_DEC       STA_ID  ...  OXBUM CHL_A PHAEO  \
0  1993-08-11T04:05:46Z     NaN     NaN  093.3 026.7  ...    NaN   NaN   NaN   
1  1993-08-11T04:06:04Z   -99.0   -99.0  093.3 026.7  ...  242.7  0.18  0.05   
2  1993-08-11T04:06:10Z   -99.0   -99.0  093.3 026.7  ...    NaN   NaN   NaN   
3  1993-08-11T04:06:11Z   -99.0   -99.0  093.3 026.7  ...  243.9  0.16  0.05   
4  1993-08-11T04:06:14Z   -99.0   -99.0  093.3 026.7  ...    NaN   NaN   NaN   

   NO3  NO2  NH4  PO4  SIL FINAL_FLAG CAST_COUNT  
0  NaN  NaN  NaN  NaN  NaN   

In [ ]:
ctd_out = ctd.drop(columns=["ORD_OCC", "EVENT_NUM"])

# Coerce all mixed-type object columns to numeric where possible
for col in ctd_out.select_dtypes(include="object").columns:
    converted = pd.to_numeric(ctd_out[col], errors="coerce")
    if converted.notna().sum() > 0:  # only replace if column has numeric data
        ctd_out[col] = converted

ctd_out.to_parquet("../Data/Parquet/downcast_all.parquet", index=False)
print("Saved to ../Data/Parquet/downcast_all.parquet")

Saved to ../Data/Parquet/downcast_all.parquet


: 

In [1]:
import pandas as pd
secchi = pd.read_csv("../Data/1949-2021_Secchi.csv", low_memory=False)

for col in secchi.select_dtypes(include="object").columns:
    converted = pd.to_numeric(secchi[col], errors="coerce")
    if converted.notna().sum() > 0:
        secchi[col] = converted

secchi.to_parquet("../Data/Parquet/1949-2021_Secchi.parquet", index=False)
print("Saved to ../Data/Parquet/1949-2021_Secchi.parquet")

Saved to ../Data/Parquet/1949-2021_Secchi.parquet


In [2]:
downcast = pd.read_parquet("../Data/Parquet/downcast_all.parquet")
secchi = pd.read_parquet("../Data/Parquet/1949-2021_Secchi.parquet")

secchi = secchi.rename(columns={"Cst_Cnt": "CAST_COUNT"})

merged = downcast.merge(secchi, on="CAST_COUNT", how="inner")

merged.to_parquet("../Data/Parquet/downcast_secchi_merged.parquet", index=False)
print(f"Merged shape: {merged.shape}")
print("Saved to ../Data/Parquet/downcast_secchi_merged.parquet")

Merged shape: (3545372, 142)
Saved to ../Data/Parquet/downcast_secchi_merged.parquet
